In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv(r"pakistan_tech_jobs_final.csv")

In [3]:
df.drop('job_description',inplace=True,axis=1)

In [4]:
df.isna().sum()
df['education']=df['education'].fillna(df['education'].mode()[0])

In [5]:
#K means to cluster job categories
#Linear to predict min max salries 
#logistic to predict job_title
#KNN to predict five related jobs
#firstly preprocessing
df['job_title']=df['job_title'].fillna(df['job_title'].mode()[0])
df.isna().sum()
df.duplicated().sum()#No Duplicates

np.int64(0)

In [6]:
df['required_skills']=df['required_skills'].str.strip()
df['required_skills']=df['required_skills'].str.lower()

In [7]:

skill_clean_map = {
    'solid programming (python': 'python',
    'or java)': 'java',
    'c/c++': 'c++',
    'rest api': 'rest apis',
    'html': 'html/css',
    'creative ui/ux': 'ui/ux design',
    'dart dazzle': 'dart',
    'devops foundations': 'devops',
    'core ml expertise': 'machine learning',
    'multi-platform jack with flutter development': 'flutter',
    'driven sqa automation': 'selenium',
    'datacraft': 'data analysis',
    'webwarp': 'web development',
    'webweave': 'web development',
    'webwizard': 'web development',
    'advance python': 'python'
}

def clean_skills(txt):
    tokens = [t.strip().lower() for t in txt.split(',') if t.strip()]
    cleaned = []
    for token in tokens:
        token = skill_clean_map.get(token, token)
        token = token.replace('(', '').replace(')', '').strip()
        if token.startswith('or '):
            token = token[3:].strip()
        if token:
            cleaned.append(token)
    return ', '.join(sorted(list(set(cleaned))))

df['required_skills'] = df['required_skills'].apply(clean_skills)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
df['role_features'] = df['job_title'] + ' ' + df['required_skills']
y = df['job_category']
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df['role_features'], y, test_size=0.25, random_state=42, stratify=y)

In [9]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

In [10]:
model1 = LogisticRegression(max_iter=1000, C=1.0)
model1.fit(X_train, y_train)
res = model1.predict(X_test)

In [11]:
print(f"Accuracy: {accuracy_score(y_test, res):.4f}")
print(f"Weighted F1: {f1_score(y_test, res, average='weighted'):.4f}\n")
print(classification_report(y_test, res))

Accuracy: 0.9254
Weighted F1: 0.9254

                        precision    recall  f1-score   support

        Cloud / DevOps       0.93      0.83      0.87        63
         Cybersecurity       1.00      0.79      0.88        28
Data Science / AI / ML       0.83      0.99      0.90       117
    Mobile Development       1.00      0.89      0.94        76
          QA / Testing       1.00      1.00      1.00        22
  Software Development       0.94      0.95      0.95       182
       Web Development       0.98      0.90      0.93        48

              accuracy                           0.93       536
             macro avg       0.95      0.91      0.93       536
          weighted avg       0.93      0.93      0.93       536



In [12]:
df.head()

,job_id,job_title,company,required_skills,city,province,experience_min,experience_max,education,employment_type,work_mode,salary_min,salary_max,posted_date,job_category,source_website,role_features
0,1001,Senior Full-Stack & AI Engineer,Jack Mitchel,"javascript, next.js, node.js, python, react, t...",Karachi,Sindh,5.0,8.0,Bachelor's,Full-time,Remote,560000.0,840000.0,2026-10-30,Data Science / AI / ML,Bano Qabil Jobs,"Senior Full-Stack & AI Engineer javascript, ne..."
1,1002,"Front end developer, UI/UX Designing",Sharp Mind,"css, html/css, java, ui/ux design",Lahore,Punjab,2.0,5.0,Bachelor's,Full-time,On-site,80000.0,110000.0,2026-10-30,Web Development,Bano Qabil Jobs,"Front end developer, UI/UX Designing css, html..."
2,1003,MACHINE LEARNING ENGINNER,MUHAMMAD AFNAN,"analytical mindset, c++, java, machine learnin...",Lahore,Punjab,2.0,5.0,Bachelor's,Full-time,On-site,60000.0,100000.0,2026-10-30,Data Science / AI / ML,Bano Qabil Jobs,"MACHINE LEARNING ENGINNER analytical mindset, ..."
3,1004,Frontend Developer (React + Next.js),VertexInvo,"next.js, react",Karachi,Sindh,2.0,5.0,Bachelor's,Full-time,On-site,50000.0,60000.0,2026-10-30,Web Development,Bano Qabil Jobs,"Frontend Developer (React + Next.js) next.js, ..."
4,1005,Mobile App Developer (Flutter),Mohsin Shiekh,"git, javascript, python, sql",Karachi,Sindh,0.0,2.0,Bachelor's,Full-time,Remote,40000.0,59998.0,2026-10-30,Mobile Development,Bano Qabil Jobs,"Mobile App Developer (Flutter) git, javascript..."
